# Chapter 13 · Making attention cheap — Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/ch13_attention.ipynb)

Reproduces **Chapter 13**: the L\* ≈ 6d crossover where attention starts to dominate FLOPs, the KV-cache league table across MHA / GQA / MQA / MLA / sliding-window, the decode read-traffic bill, and a working GQA implementation verified against full attention.

📖 Read the chapter: [Making attention cheap](https://omar-florez.github.io/training-efficient-llms/pdf/ch13_attention.pdf) · 🎛 [The Attention Zoo](https://omar-florez.github.io/training-efficient-llms/interactive/attention.html)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
print("ready")

## 1 · Which bill are you paying? The L\* = 6d crossover

Per token per layer: attention's pairwise term costs 4·L·d and grows with context; projections + SwiGLU cost ≈ 24·d² and don't. Below the crossing, attention is a rounding error; above it, it *is* the model.

In [ ]:
d, d_ff = 4096, 11008
L = np.arange(256, 131072, 256)
attn_flops  = 4 * L * d
other_flops = np.full_like(L, 8*d*d + 6*d*d_ff)
Lstar = (8*d*d + 6*d*d_ff) / (4*d)

plt.figure(figsize=(8.5, 4.2))
plt.plot(L/1000, attn_flops/1e6, color=NAVY, lw=2, label="attention pairs: 4·L·d")
plt.plot(L/1000, other_flops/1e6, color=BLUE, lw=2, label="projections + MLP: ≈24·d² (flat)")
plt.axvline(Lstar/1000, color=AMBER, ls="--"); plt.annotate(f"L* ≈ 6d ≈ {Lstar/1000:.0f}K tokens", (Lstar/1000+2, 100), color=AMBER)
plt.xlabel("context length (K tokens)"); plt.ylabel("MFLOPs per token per layer")
plt.legend(); plt.title("7B geometry: attention overtakes everything else near 25K context")
plt.show()

share = attn_flops / (attn_flops + other_flops)
for ctx in [4096, 25000, 131072]:
    idx = np.argmin(abs(L-ctx))
    print(f"at {ctx:>7,} context: attention = {share[idx]*100:4.0f}% of per-token FLOPs")

## 2 · The KV-cache league table

Per token, all layers, BF16 — the numbers behind Table 13.1 and the Attention Zoo.

In [ ]:
variants = {
  "MHA (Llama 2 7B)":      2*32*32*128*2,
  "GQA-8 (Llama 3 8B)":    2*32*8*128*2,
  "GQA-4 (Qwen2.5-7B)":    2*28*4*128*2,
  "MQA (PaLM-style 7B)":   2*32*1*128*2,
  "MLA (DeepSeek-V3 671B)":576*2*61,
}
names, vals = list(variants), np.array(list(variants.values()))
plt.figure(figsize=(9, 3.8))
bars = plt.barh(names[::-1], vals[::-1]/1024, color=[AMBER, LIGHT, BLUE, BLUE, NAVY])
for b, v in zip(bars, vals[::-1]):
    plt.text(v/1024 + 8, b.get_y()+b.get_height()/2, f"{v/1024:,.0f} KB", va="center", fontsize=10)
plt.xlabel("KB of cache per generated token (all layers, BF16)")
plt.title("A 671B model that caches less per token than a vanilla 7B")
plt.tight_layout(); plt.show()

## 3 · Decode is a memory bill: bytes read per token

Every generated token re-reads the whole cache. Sliding windows change the *shape* of the curve, not just its height — the cache stops growing at W.

In [ ]:
ctx = np.arange(0, 65536, 512)
def cache_bytes(per_tok, cap=None):
    eff = np.minimum(ctx, cap) if cap else ctx
    return per_tok * eff / 1e9

plt.figure(figsize=(8.5, 4.2))
plt.plot(ctx/1000, cache_bytes(variants["MHA (Llama 2 7B)"]),   color=NAVY, lw=2, label="MHA")
plt.plot(ctx/1000, cache_bytes(variants["GQA-8 (Llama 3 8B)"]), color=BLUE, lw=2, label="GQA-8")
plt.plot(ctx/1000, cache_bytes(variants["GQA-8 (Llama 3 8B)"], cap=4096), color=AMBER, lw=2, label="GQA-8 + sliding window 4K (Mistral)")
plt.plot(ctx/1000, cache_bytes(variants["MLA (DeepSeek-V3 671B)"]), color=LIGHT, lw=2, label="MLA (671B model!)")
plt.xlabel("context length (K tokens)"); plt.ylabel("GB read per generated token")
plt.legend(); plt.title("The bytes every single generated token must stream from HBM")
plt.show()
mha100 = variants["MHA (Llama 2 7B)"] * 32768 * 100 / 1e9
print(f"generate 100 tokens at 32K context, MHA 7B: {mha100:,.0f} GB read from memory")

## 4 · GQA in twelve lines, verified

Grouped-query attention = full attention where KV heads are *shared* within groups. Implemented with `repeat_interleave`; when groups = heads it must equal MHA exactly. *(Needs PyTorch — preinstalled on Colab.)*

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(0)

def gqa(q, k, v, h_kv):
    """q: (B, h, L, dh) · k,v: (B, h_kv, L, dh) — share each KV head across h//h_kv queries."""
    h = q.shape[1]
    k = k.repeat_interleave(h // h_kv, dim=1)
    v = v.repeat_interleave(h // h_kv, dim=1)
    L = q.shape[2]
    scores = q @ k.transpose(-2, -1) / q.shape[-1] ** 0.5
    scores = scores.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool), 1), float("-inf"))
    return scores.softmax(-1) @ v

B, h, Lq, dh = 2, 8, 16, 32
q = torch.randn(B, h, Lq, dh); k = torch.randn(B, h, Lq, dh); v = torch.randn(B, h, Lq, dh)

full = gqa(q, k, v, h_kv=8)                       # 8 KV heads = plain MHA
print("GQA(h_kv=h) == MHA:", torch.allclose(full, gqa(q, k, v, 8)))

k2, v2 = k[:, ::4].contiguous(), v[:, ::4].contiguous()   # keep 2 of 8 KV heads
out2 = gqa(q, k2, v2, h_kv=2)
print("GQA(h_kv=2) output:", tuple(out2.shape), " cache reduction: 4×")
print("outputs differ from MHA (as they should):", not torch.allclose(full, out2))

## Break things

1. In §1, redo the crossover for GPT-2 small (d=768, d_ff=3072, 2-matrix MLP). Why do small models feel long context so much sooner?
2. In §2, add a hypothetical MLA for the 7B: latent 512+64, 32 layers. Competitive with MQA?
3. In §3, model Gemma 3's stack: 5 of every 6 layers capped at W=1024, one full. Plot the blended curve.
4. In §4, time `F.scaled_dot_product_attention` vs the naive version at L = 2048 on a Colab GPU — you're timing FlashAttention (§13.2).